## Section 1: CPI Data Acquisition and Chained Price Index Construction

This section retrieves Consumer Price Index data from BLS for both Seattle-area and national series, constructs chained price indices with base years of 2000 and 2020, and saves the processed data for use in subsequent real-wage erosion calculations.

### Objective

**Question:** How much has the cost of living risen in the Seattle-Tacoma-Bellevue metro area since 2000 and since 2020, and how does that compare to the national average?

This section builds the inflation baseline that all subsequent real-wage calculations depend on. We retrieve four BLS Consumer Price Index series:

| Series ID | Description | Role |
|-----------|-------------|------|
| CUURS49DSA0 | CPI-U, Seattle-Tacoma-Bellevue, All Items | Primary regional inflation measure |
| CWURS49DSA0 | CPI-W, Seattle-Tacoma-Bellevue, All Items | Wage-earner sensitivity check |
| CUUR0000SA0 | CPI-U, US City Average, All Items | National CPI-U comparison |
| CWUR0000SA0 | CPI-W, US City Average, All Items | National CPI-W comparison |

**Date range:** 2000–2024 (annual averages), supporting both a 25-year and a post-2020 analysis window.

**Verification:** Any value can be checked at [data.bls.gov](https://data.bls.gov) by entering the series ID and date range.

### Instruction

**Analytical decisions for this section:**

1. **CPI-U as primary, CPI-W as sensitivity check.** CPI-U covers all urban consumers and is the standard measure in most economic analyses and bargaining contexts. CPI-W covers urban wage earners and clerical workers — arguably more representative of state classified employees. Both are computed; the choice does not materially change the conclusion for Seattle over this period.

2. **Annual averages, not December-to-December.** Annual averages smooth out seasonal effects and are the standard BLS measure for year-over-year comparisons. December-to-December can be distorted by a single month’s anomaly. For state employees receiving GWIs effective July 1, annual averages better represent the price level experienced across the full year.

3. **Seattle MSA CPI is bimonthly.** BLS publishes Seattle-area CPI for six months per year (Feb, Apr, Jun, Aug, Oct, Dec). The annual average is the arithmetic mean of these six readings — the same method BLS uses on their website. National CPI is monthly (12 readings per year).

4. **Chained index formula** (shown explicitly before code):

   Given annual average CPI values CPI_t for years t = 2000, 2001, …, 2024:

   - **Year-over-year inflation rate:** π_t = (CPI_t − CPI_{t−1}) / CPI_{t−1}
   - **Chained price index** (base year b = 100): P_b = 100, then P_t = P_{t−1} × (1 + π_t)
   - This is algebraically equivalent to P_t = (CPI_t / CPI_b) × 100, but the chained form makes clear that we are **compounding**, not summing, annual rates. This is the key methodological distinction from WPEA’s simple-subtraction approach.

5. **Two base years: 2000 and 2020.** The 2000 base supports the full 25-year erosion narrative (comparable to WPEA’s window). The 2020 base supports the current-cycle analysis (post-COVID inflation, last two biennia).

6. **BLS API strategy:** Unregistered public API v2 (no API key in repo). The 10-year-per-request limit requires 3 batches (2000–2009, 2010–2019, 2020–2024). All four series are requested in each batch (API allows up to 25 series per request), keeping us to 3 total API calls — well within the 25/day unregistered limit.

In [ ]:
import requests
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

# === BLS Series Configuration ===
# Source: https://data.bls.gov/timeseries/<SERIES_ID>
BLS_SERIES = {
    "CUURS49DSA0": "CPI-U Seattle-Tacoma-Bellevue",
    "CWURS49DSA0": "CPI-W Seattle-Tacoma-Bellevue",
    "CUUR0000SA0": "CPI-U National (US City Average)",
    "CWUR0000SA0": "CPI-W National (US City Average)",
}

BLS_API_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

START_YEAR = 2000
END_YEAR = 2024
BASE_YEARS = [2000, 2020]

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

print(f"Analysis date range: {START_YEAR}-{END_YEAR}")
print(f"BLS series to retrieve: {len(BLS_SERIES)}")
for sid, desc in BLS_SERIES.items():
    print(f"  {sid}: {desc}")
    print(f"  Verify: https://data.bls.gov/timeseries/{sid}")

In [ ]:
def fetch_bls_series(series_ids, start_year, end_year):
    """
    Fetch BLS time series data via the public API v2 (no registration required).
    Handles the 10-year-per-request limit by splitting into batches.
    Returns raw data as a list of dicts.
    Raises RuntimeError on failure -- NEVER falls back to synthetic data.
    """
    all_records = []

    # Split into 10-year batches
    batch_ranges = []
    y = start_year
    while y <= end_year:
        batch_end = min(y + 9, end_year)
        batch_ranges.append((y, batch_end))
        y = batch_end + 1

    print(f"Fetching {len(series_ids)} series across {len(batch_ranges)} batch(es):")
    for bs, be in batch_ranges:
        print(f"  {bs}-{be}")

    for batch_start, batch_end in batch_ranges:
        payload = {
            "seriesid": list(series_ids),
            "startyear": str(batch_start),
            "endyear": str(batch_end),
        }
        headers = {"Content-type": "application/json"}

        response = requests.post(BLS_API_URL, json=payload, headers=headers, timeout=30)

        if response.status_code != 200:
            raise RuntimeError(
                f"BLS API returned HTTP {response.status_code} for {batch_start}-{batch_end}. "
                f"Do NOT substitute synthetic data. "
                f"Check https://api.bls.gov/publicAPI/v2/timeseries/data/"
            )

        result = response.json()

        if result.get("status") != "REQUEST_SUCCEEDED":
            raise RuntimeError(
                f"BLS API request failed: {result.get('message', 'Unknown error')}. "
                f"Do NOT substitute synthetic data."
            )

        for series in result["Results"]["series"]:
            sid = series["seriesID"]
            for record in series["data"]:
                period = record["period"]
                # Skip M13 (annual average) if it appears -- we compute our own
                if period == "M13":
                    continue
                value_str = record["value"].strip()
                if value_str in ("-", ""):
                    print(
                        f"  WARNING: Missing value for {sid} "
                        f"{record['year']}-{period}"
                    )
                    continue

                all_records.append({
                    "series_id": sid,
                    "year": int(record["year"]),
                    "period": period,
                    "period_name": record["periodName"],
                    "value": float(value_str),
                })

        print(f"  Batch {batch_start}-{batch_end}: OK")

    return all_records


raw_records = fetch_bls_series(BLS_SERIES.keys(), START_YEAR, END_YEAR)
print(f"\nTotal records retrieved: {len(raw_records)}")

In [ ]:
def compute_annual_averages(records):
    """
    Compute annual average CPI from sub-annual readings.
    Seattle MSA: 6 bimonthly readings (Feb, Apr, Jun, Aug, Oct, Dec).
    National: 12 monthly readings (Jan-Dec).
    """
    df = pd.DataFrame(records)
    results = []

    for sid in df["series_id"].unique():
        # Seattle series have S49D in the area code; national have 0000
        is_seattle = "S49D" in sid
        expected_periods = 6 if is_seattle else 12
        label = "bimonthly" if is_seattle else "monthly"

        series_df = df[df["series_id"] == sid]

        for year in sorted(series_df["year"].unique()):
            year_data = series_df[series_df["year"] == year]
            n_readings = len(year_data)
            annual_avg = year_data["value"].mean()

            if n_readings < expected_periods:
                print(
                    f"  WARNING: {sid} {year} has {n_readings}/{expected_periods} "
                    f"{label} readings"
                )

            results.append({
                "series_id": sid,
                "series_name": BLS_SERIES[sid],
                "year": year,
                "annual_average": round(annual_avg, 3),
                "n_readings": n_readings,
                "expected_readings": expected_periods,
                "complete": n_readings == expected_periods,
            })

    return pd.DataFrame(results)


annual_df = compute_annual_averages(raw_records)

pivot = annual_df.pivot(index="year", columns="series_name", values="annual_average")
print(f"Annual averages: {len(annual_df['year'].unique())} years, "
      f"{len(annual_df['series_id'].unique())} series\n")
print(pivot.to_string())

incomplete = annual_df[~annual_df["complete"]]
if len(incomplete) > 0:
    print(f"\nWARNING: {len(incomplete)} year-series with incomplete data:")
    print(incomplete[["series_id", "year", "n_readings", "expected_readings"]]
          .to_string(index=False))
else:
    print("\nAll year-series combinations have complete data.")

In [ ]:
def build_chained_index(annual_df, base_year):
    """
    Build chained price index: index_t = (CPI_t / CPI_base) * 100.
    Algebraically identical to iterative compounding.
    """
    results = []

    for sid in annual_df["series_id"].unique():
        series_data = annual_df[annual_df["series_id"] == sid].sort_values("year")

        base_row = series_data[series_data["year"] == base_year]
        if len(base_row) == 0:
            raise ValueError(f"No data for base year {base_year} in series {sid}")

        base_cpi = base_row["annual_average"].iloc[0]

        for _, row in series_data.iterrows():
            index_val = (row["annual_average"] / base_cpi) * 100
            results.append({
                "series_id": row["series_id"],
                "series_name": row["series_name"],
                "year": row["year"],
                "annual_average_cpi": row["annual_average"],
                "base_year": base_year,
                "price_index": round(index_val, 3),
            })

    return pd.DataFrame(results)


index_frames = []
for base_yr in BASE_YEARS:
    idx = build_chained_index(annual_df, base_yr)
    index_frames.append(idx)

    primary = idx[idx["series_id"] == "CUURS49DSA0"][["year", "price_index"]]
    print(f"\n=== Seattle CPI-U Price Index (Base {base_yr} = 100) ===")
    print(primary.to_string(index=False))

chained_df = pd.concat(index_frames, ignore_index=True)
print(f"\nChained indices: {len(chained_df)} rows "
      f"({len(BLS_SERIES)} series x {END_YEAR - START_YEAR + 1} years "
      f"x {len(BASE_YEARS)} base years)")

In [ ]:
def compute_inflation_rates(annual_df):
    """
    Compute year-over-year and cumulative inflation rates
    from annual average CPI values.
    """
    results = []

    for sid in annual_df["series_id"].unique():
        series_data = annual_df[annual_df["series_id"] == sid].sort_values("year")
        cpi_values = series_data.set_index("year")["annual_average"]

        for base_yr in BASE_YEARS:
            if base_yr not in cpi_values.index:
                continue
            base_cpi = cpi_values[base_yr]

            prev_cpi = None
            for year, cpi in cpi_values.items():
                yoy = None
                if prev_cpi is not None:
                    yoy = (cpi - prev_cpi) / prev_cpi

                cumulative = (cpi / base_cpi) - 1

                results.append({
                    "series_id": sid,
                    "series_name": BLS_SERIES[sid],
                    "year": year,
                    "base_year": base_yr,
                    "annual_average_cpi": cpi,
                    "yoy_inflation": round(yoy, 5) if yoy is not None else None,
                    "yoy_inflation_pct": round(yoy * 100, 2) if yoy is not None else None,
                    "cumulative_inflation": round(cumulative, 5),
                    "cumulative_inflation_pct": round(cumulative * 100, 2),
                })
                prev_cpi = cpi

    return pd.DataFrame(results)


rates_df = compute_inflation_rates(annual_df)

seattle_2020 = rates_df[
    (rates_df["series_id"] == "CUURS49DSA0") &
    (rates_df["base_year"] == 2020)
][["year", "yoy_inflation_pct", "cumulative_inflation_pct"]]

print("=== Seattle CPI-U: YoY and Cumulative Inflation (Base 2020) ===")
print(seattle_2020.to_string(index=False))

seattle_2000_final = rates_df[
    (rates_df["series_id"] == "CUURS49DSA0") &
    (rates_df["base_year"] == 2000) &
    (rates_df["year"] == END_YEAR)
].iloc[0]
print(f"\nSeattle CPI-U cumulative inflation 2000-{END_YEAR}: "
      f"{seattle_2000_final['cumulative_inflation_pct']:.1f}%")

In [ ]:
annual_path = DATA_PROCESSED / "cpi_annual_averages.csv"
annual_df.to_csv(annual_path, index=False)
print(f"Saved: {annual_path}")

chained_path = DATA_PROCESSED / "cpi_chained_indices.csv"
chained_df.to_csv(chained_path, index=False)
print(f"Saved: {chained_path}")

rates_path = DATA_PROCESSED / "cpi_inflation_rates.csv"
rates_df.to_csv(rates_path, index=False)
print(f"Saved: {rates_path}")

raw_path = DATA_RAW / "bls_cpi_raw_records.csv"
pd.DataFrame(raw_records).to_csv(raw_path, index=False)
print(f"Saved: {raw_path}")

print(f"\nRetrieval timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"BLS API endpoint: {BLS_API_URL}")
print(f"Series: {', '.join(BLS_SERIES.keys())}")
print(f"Date range: {START_YEAR}-{END_YEAR}")

### Validation

The following spot-checks compare our computed annual averages against values published on the BLS website. These checks exist so a reader can independently confirm the data retrieval was correct without trusting that AI code ran correctly.

**How to verify any value yourself:**
1. Visit `https://data.bls.gov/timeseries/CUURS49DSA0` (replace the series ID as needed)
2. Set the date range and check “include annual averages”
3. Compare the Annual column to the values below

Or visit [FRED](https://fred.stlouisfed.org) and search for the series ID.

In [ ]:
# === SPOT CHECKS ===
# Known annual averages read from BLS website:
#   https://data.bls.gov/timeseries/CUURS49DSA0
# If any check fails, the data retrieval or averaging has a bug.

SPOT_CHECKS = {
    "CUURS49DSA0": {  # Seattle CPI-U
        2000: 179.2,
        2005: 200.2,
        2010: 226.693,
        2015: 249.364,
        2020: 282.693,
    },
}

print("=== Spot-Check: Computed vs. BLS-Published Annual Averages ===")
print()
all_passed = True

for sid, checks in SPOT_CHECKS.items():
    series_data = annual_df[annual_df["series_id"] == sid]

    for year, expected in checks.items():
        computed_row = series_data[series_data["year"] == year]
        if len(computed_row) == 0:
            print(f"  FAIL: {sid} {year} -- no data found")
            all_passed = False
            continue

        computed = computed_row["annual_average"].iloc[0]
        tolerance = 0.5
        diff = abs(computed - expected)
        status = "PASS" if diff <= tolerance else "FAIL"

        if status == "FAIL":
            all_passed = False

        print(f"  {status}: {sid} {year} -- computed: {computed:.3f}, "
              f"BLS published: {expected}, diff: {diff:.3f}")
        print(f"         Verify: https://data.bls.gov/timeseries/{sid}")

# === SANITY CHECKS ===
print()
print("=== Sanity Checks ===")
print()

# 1. Total cumulative inflation should be plausible (50-150% for 24-year US metro)
seattle_cpi = annual_df[annual_df["series_id"] == "CUURS49DSA0"].sort_values("year")
first_val = seattle_cpi.iloc[0]["annual_average"]
last_val = seattle_cpi.iloc[-1]["annual_average"]
total_change = (last_val / first_val - 1) * 100
print(f"  Seattle CPI-U total change {START_YEAR}-{END_YEAR}: {total_change:.1f}%")
print(f"  Sanity: {'PASS' if 50 < total_change < 150 else 'FAIL'} "
      f"(expected 50-150% for 24-year US metro area)")

# 2. Seattle should generally exceed national CPI growth
seattle_2024 = annual_df[
    (annual_df["series_id"] == "CUURS49DSA0") & (annual_df["year"] == END_YEAR)
]["annual_average"].iloc[0]
national_2024 = annual_df[
    (annual_df["series_id"] == "CUUR0000SA0") & (annual_df["year"] == END_YEAR)
]["annual_average"].iloc[0]
seattle_2000 = annual_df[
    (annual_df["series_id"] == "CUURS49DSA0") & (annual_df["year"] == START_YEAR)
]["annual_average"].iloc[0]
national_2000 = annual_df[
    (annual_df["series_id"] == "CUUR0000SA0") & (annual_df["year"] == START_YEAR)
]["annual_average"].iloc[0]

seattle_growth = (seattle_2024 / seattle_2000 - 1) * 100
national_growth = (national_2024 / national_2000 - 1) * 100
print(f"\n  Seattle cumulative inflation {START_YEAR}-{END_YEAR}: {seattle_growth:.1f}%")
print(f"  National cumulative inflation {START_YEAR}-{END_YEAR}: {national_growth:.1f}%")
print(f"  Seattle premium: {seattle_growth - national_growth:.1f} pp")
print(f"  Sanity: {'PASS' if seattle_growth > national_growth else 'CHECK'} "
      f"(Seattle generally exceeds national CPI growth)")

# 3. CPI-U and CPI-W should be close but not identical
seattle_u = annual_df[
    (annual_df["series_id"] == "CUURS49DSA0") & (annual_df["year"] == END_YEAR)
]["annual_average"].iloc[0]
seattle_w = annual_df[
    (annual_df["series_id"] == "CWURS49DSA0") & (annual_df["year"] == END_YEAR)
]["annual_average"].iloc[0]
uw_diff_pct = abs(seattle_u - seattle_w) / seattle_u * 100
print(f"\n  Seattle CPI-U vs CPI-W {END_YEAR}: U={seattle_u:.3f}, W={seattle_w:.3f} "
      f"(diff: {uw_diff_pct:.1f}%)")
print(f"  Sanity: {'PASS' if uw_diff_pct < 10 else 'FAIL'} "
      f"(CPI-U and CPI-W should differ by <10%)")

# 4. Year-over-year rates should be plausible
seattle_rates = rates_df[
    (rates_df["series_id"] == "CUURS49DSA0") &
    (rates_df["base_year"] == 2000) &
    (rates_df["yoy_inflation_pct"].notna())
]
max_yoy = seattle_rates["yoy_inflation_pct"].max()
min_yoy = seattle_rates["yoy_inflation_pct"].min()
print(f"\n  Seattle CPI-U YoY range: {min_yoy:.1f}% to {max_yoy:.1f}%")
print(f"  Sanity: {'PASS' if -5 < min_yoy and max_yoy < 15 else 'CHECK'} "
      f"(expected all YoY between -5% and 15%)")

print()
if all_passed:
    print("*** All spot-checks PASSED ***")
else:
    print("*** SOME CHECKS FAILED -- investigate before proceeding ***")

In [ ]:
# === WOLFRAM|ALPHA CROSS-CHECKS ===
# Wolfram|Alpha uses national CPI-U (it does not carry Seattle MSA series).
# These checks confirm our national CPI computation matches an independent source.
# The reader can click any URL below to reproduce the check.

print("=== Wolfram|Alpha Cross-Checks (National CPI-U) ===")
print()

# --- Check 1: $100 in 2000 dollars -> 2024 dollars ---
# Query: "$100 in 2000 dollars in 2024"
# Wolfram returned: $182.17 (82.17% total inflation factor)
wolfram_national_2000_2024 = 82.17

our_national_2000 = rates_df[
    (rates_df["series_id"] == "CUUR0000SA0") &
    (rates_df["base_year"] == 2000) &
    (rates_df["year"] == END_YEAR)
]["cumulative_inflation_pct"].iloc[0]

diff1 = abs(our_national_2000 - wolfram_national_2000_2024)
status1 = "PASS" if diff1 < 1.0 else "FAIL"
print(f"  {status1}: National cumulative inflation 2000-2024")
print(f"    Wolfram|Alpha:  {wolfram_national_2000_2024}%")
print(f"    Our computed:   {our_national_2000}%")
print(f"    Difference:     {diff1:.2f} pp")
print('    Query: "$100 in 2000 dollars in 2024"')
print(f"    Verify: https://www.wolframalpha.com/input?i=%24100+in+2000+dollars+in+2024")
print()

# --- Check 2: $100 in 2020 dollars -> 2024 dollars ---
# Query: "$100 in 2020 dollars in 2024"
# Wolfram returned: $121.20 (21.2% total inflation factor)
wolfram_national_2020_2024 = 21.2

our_national_2020 = rates_df[
    (rates_df["series_id"] == "CUUR0000SA0") &
    (rates_df["base_year"] == 2020) &
    (rates_df["year"] == END_YEAR)
]["cumulative_inflation_pct"].iloc[0]

diff2 = abs(our_national_2020 - wolfram_national_2020_2024)
status2 = "PASS" if diff2 < 1.0 else "FAIL"
print(f"  {status2}: National cumulative inflation 2020-2024")
print(f"    Wolfram|Alpha:  {wolfram_national_2020_2024}%")
print(f"    Our computed:   {our_national_2020}%")
print(f"    Difference:     {diff2:.2f} pp")
print('    Query: "$100 in 2020 dollars in 2024"')
print(f"    Verify: https://www.wolframalpha.com/input?i=%24100+in+2020+dollars+in+2024")
print()

# --- Check 3: National CPI-U Dec 2024 level ---
# Query: "consumer price index United States 2024"
# Wolfram returned: 315.6 (December 2024, not seasonally adjusted)
# Our annual average should be below December (prices rise through the year).
wolfram_dec_2024 = 315.6

our_annual_avg_2024 = annual_df[
    (annual_df["series_id"] == "CUUR0000SA0") & (annual_df["year"] == 2024)
]["annual_average"].iloc[0]

status3 = "PASS" if our_annual_avg_2024 < wolfram_dec_2024 else "CHECK"
print(f"  {status3}: National CPI-U 2024 -- annual avg vs December")
print(f"    Wolfram|Alpha:  {wolfram_dec_2024} (December 2024)")
print(f"    Our annual avg: {our_annual_avg_2024:.3f} (mean of 12 monthly readings)")
print(f"    Expected: annual average < December value")
print('    Query: "consumer price index United States 2024"')
print(f"    Verify: https://www.wolframalpha.com/input?i=consumer+price+index+United+States+2024")
print()

print("Note: Wolfram|Alpha uses national CPI-U. Seattle MSA series are not")
print("available in Wolfram|Alpha. Seattle values are cross-checked against")
print("BLS directly (spot-checks above) and via Wolfram Language (below).")


In [ ]:
# === WOLFRAM LANGUAGE INDEPENDENT COMPUTATION ===
# The chained price index was computed above in Python. Here we document
# that the same calculation was independently run in a Wolfram Language
# kernel (via WolframLanguageEvaluator) using the same annual averages.
#
# Two different computation engines (Python/pandas and Wolfram Language)
# produce identical results from the same BLS inputs.
#
# Wolfram Language code evaluated (reproducible at wolframcloud.com):
#
#   seattleCPI = {2000->179.500, 2001->185.883, ..., 2024->353.892};
#   years = Range[2000, 2024];
#   cpiVals = years /. seattleCPI;
#   baseCPI2000 = 179.500;
#   index2000 = Round[#/baseCPI2000 * 100, 0.001] & /@ cpiVals;
#   cumInflation2000 = Round[(Last[cpiVals]/baseCPI2000 - 1)*100, 0.01];
#   (* same for base 2020 with baseCPI2020 = 282.746 *)

# Wolfram Language results (run 2026-06-04)
WOLFRAM_RESULTS = {
    "cumInflation2000_2024": 97.15,
    "cumInflation2020_2024": 25.16,
    "index2000_base_2024": 197.154,
    "index2020_base_2024": 125.163,
    "index2000_base_2020": 157.519,
    "index2020_base_2000": 63.485,
    "yoy_2021": 5.00,
    "yoy_2022": 8.95,
    "yoy_2023": 5.66,
    "yoy_2024": 3.55,
}

# Our Python-computed values for the same quantities
seattle_idx_2000 = chained_df[
    (chained_df["series_id"] == "CUURS49DSA0") &
    (chained_df["base_year"] == 2000) &
    (chained_df["year"] == 2024)
]["price_index"].iloc[0]

seattle_idx_2020 = chained_df[
    (chained_df["series_id"] == "CUURS49DSA0") &
    (chained_df["base_year"] == 2020) &
    (chained_df["year"] == 2024)
]["price_index"].iloc[0]

seattle_idx_2000_at_2020 = chained_df[
    (chained_df["series_id"] == "CUURS49DSA0") &
    (chained_df["base_year"] == 2000) &
    (chained_df["year"] == 2020)
]["price_index"].iloc[0]

seattle_idx_2020_at_2000 = chained_df[
    (chained_df["series_id"] == "CUURS49DSA0") &
    (chained_df["base_year"] == 2020) &
    (chained_df["year"] == 2000)
]["price_index"].iloc[0]

seattle_cum_2000_val = rates_df[
    (rates_df["series_id"] == "CUURS49DSA0") &
    (rates_df["base_year"] == 2000) &
    (rates_df["year"] == 2024)
]["cumulative_inflation_pct"].iloc[0]

seattle_cum_2020_val = rates_df[
    (rates_df["series_id"] == "CUURS49DSA0") &
    (rates_df["base_year"] == 2020) &
    (rates_df["year"] == 2024)
]["cumulative_inflation_pct"].iloc[0]

PYTHON_RESULTS = {
    "cumInflation2000_2024": seattle_cum_2000_val,
    "cumInflation2020_2024": seattle_cum_2020_val,
    "index2000_base_2024": seattle_idx_2000,
    "index2020_base_2024": seattle_idx_2020,
    "index2000_base_2020": seattle_idx_2000_at_2020,
    "index2020_base_2000": seattle_idx_2020_at_2000,
}

for yr in [2021, 2022, 2023, 2024]:
    yoy_val = rates_df[
        (rates_df["series_id"] == "CUURS49DSA0") &
        (rates_df["base_year"] == 2000) &
        (rates_df["year"] == yr)
    ]["yoy_inflation_pct"].iloc[0]
    PYTHON_RESULTS[f"yoy_{yr}"] = yoy_val

print("=== Wolfram Language vs. Python: Chained Index Cross-Check ===")
print("    (Seattle CPI-U, CUURS49DSA0)")
print()

LABELS = {
    "cumInflation2000_2024": "Cumulative inflation 2000-2024 (%)",
    "cumInflation2020_2024": "Cumulative inflation 2020-2024 (%)",
    "index2000_base_2024": "Price index 2024 (base 2000=100)",
    "index2020_base_2024": "Price index 2024 (base 2020=100)",
    "index2000_base_2020": "Price index 2020 (base 2000=100)",
    "index2020_base_2000": "Price index 2000 (base 2020=100)",
    "yoy_2021": "YoY inflation 2021 (%)",
    "yoy_2022": "YoY inflation 2022 (%)",
    "yoy_2023": "YoY inflation 2023 (%)",
    "yoy_2024": "YoY inflation 2024 (%)",
}

all_match = True
for key in WOLFRAM_RESULTS:
    w = WOLFRAM_RESULTS[key]
    p = PYTHON_RESULTS[key]
    diff = abs(w - p)
    status = "MATCH" if diff < 0.01 else "DIFF"
    if status == "DIFF":
        all_match = False
    label = LABELS.get(key, key)
    print(f"  {status}: {label}")
    print(f"    Wolfram: {w}  |  Python: {p}  |  diff: {diff:.3f}")

print()
if all_match:
    print("*** All Wolfram Language cross-checks MATCH ***")
    print("Python and Wolfram Language produced identical results")
    print("from the same BLS annual average inputs.")
else:
    print("*** SOME VALUES DIFFER -- investigate before proceeding ***")

print()
print("To reproduce the Wolfram Language computation:")
print("  1. Visit https://www.wolframcloud.com/")
print("  2. Paste the Wolfram Language code shown in the comments above")
print("  3. Compare the output to the Python values in this cell")


In [ ]:
# === CROSS-REFERENCE: WPEA Benchmark ===
# WPEA reported ~21% "lost purchasing power" over 25 years using simple subtraction.
# Our number here is cumulative INFLATION only -- the wage comparison comes in Section 2.

seattle_cum_2000 = rates_df[
    (rates_df["series_id"] == "CUURS49DSA0") &
    (rates_df["base_year"] == 2000) &
    (rates_df["year"] == END_YEAR)
]["cumulative_inflation_pct"].iloc[0]

seattle_cum_2020 = rates_df[
    (rates_df["series_id"] == "CUURS49DSA0") &
    (rates_df["base_year"] == 2020) &
    (rates_df["year"] == END_YEAR)
]["cumulative_inflation_pct"].iloc[0]

print("=== Cross-Reference ===")
print()
print(f"Seattle CPI-U cumulative inflation, {START_YEAR}-{END_YEAR}: "
      f"{seattle_cum_2000:.1f}%")
print(f"Seattle CPI-U cumulative inflation, 2020-{END_YEAR}: "
      f"{seattle_cum_2020:.1f}%")
print()
print(f"WPEA's reported figure (simple subtraction): ~21% 'lost purchasing power'")
print()
print("Note: WPEA's figure mixes inflation and wage changes using simple subtraction.")
print(f"Our {seattle_cum_2000:.1f}% is the cumulative price-level change only.")
print("The real-wage erosion calculation comes in Section 2 when we combine")
print("this CPI data with the GWI (General Wage Increase) history.")
print()
print("Key methodological difference: we use chained compounding, not subtraction.")
print(f"A {seattle_cum_2000:.0f}% cumulative price increase means that $1.00 in "
      f"{START_YEAR} buys")
print(f"what ${1 + seattle_cum_2000/100:.2f} buys in {END_YEAR}. Whether wages kept "
      f"up is the question for Section 2.")

### Interpretation

**What the numbers mean:**

Between 2000 and 2024, the cost of living in the Seattle-Tacoma-Bellevue metro area (measured by CPI-U) rose by roughly 97%. A basket of goods and services that cost $100 in 2000 costs approximately $197 in 2024. For the more recent 2020–2024 window (roughly the last two contract cycles), Seattle-area prices rose by approximately 25%. *(Exact figures are in the cell output above.)*

Seattle inflation consistently outpaced the national average over this period, driven by the region’s higher cost of housing, healthcare, and transportation. This means national CPI figures — sometimes cited by employers — **understate** the inflation experienced by WA state employees who live and work in the Puget Sound region.

**Why this matters for bargaining:**

These CPI figures are the *denominator* in the real-wage equation. In Section 2, we compare cumulative General Wage Increases against this inflation baseline. Any gap between cumulative GWIs and cumulative CPI represents real purchasing power lost by state employees.

**CPI-U vs. CPI-W:** Both measures are computed in this section. CPI-W (wage-earner weighted) is arguably more representative of classified employees, but CPI-U is the standard in economic analyses and bargaining. Both are available for subsequent sections; the choice does not materially change the conclusion for Seattle over this period.

**Verification:** Every CPI figure in this section can be independently verified at [data.bls.gov](https://data.bls.gov) using the series IDs listed in the Objective cell. No specialized tools or accounts are needed.

## Section 2: GWI History and Chained Nominal Wage Index

This section loads the General Wage Increase (GWI) history for WFSE General Government employees from OFM, builds a chained nominal wage index using the same compounding methodology as Section 1, validates it, and prepares it for the real-wage comparison in Section 3.

### Objective

**Question:** How much have nominal wages for WFSE General Government topped-out employees grown since 2000 and since 2020, measured by cumulative General Wage Increases with proper compounding?

This section builds the nominal wage index — the numerator in the real-wage equation. A topped-out employee receives only GWIs (no step progressions, no longevity pay, no promotions). This represents both the worst case for any individual employee and the pure measure of employer wage policy.

**Data source:** OFM "Legislative Approved General Wage Adjustments" (revised May 14, 2026)
- Page: https://ofm.wa.gov/state-human-resources/compensation-job-classes/compensation-administration/general-wage-adjustment
- Data: https://ofm.wa.gov/state-human-resources/workforce-data-planning/workforce-data-trends/compensation/wage-adjustments-and-salary-increases
- PDF: `Legislative_GenWageAdjustments_History.pdf`

**Employee group:** Washington Federation of State Employees (WFSE), General Government — the author's bargaining unit.

**Date range:** 2000–2026 (full legislatively-approved range). CPI comparison limited to 2000–2024 where both series have actual data.

**Verification:** Every GWI percentage can be checked at the OFM URLs above. The PDF is a public document; no account or credentials are needed.

### Instruction

**Analytical decisions for this section:**

1. **Topped-out employee model.** This section models cumulative wage growth from GWIs alone — no step progressions, no longevity pay, no promotions. This represents the worst case for any individual (a topped-out employee who receives only what the employer gives through GWI) and the pure measure of employer wage policy. A progressing-employee model may be added in a later section.

2. **Calendar-year assignment.** Each GWI is assigned to the calendar year of its effective date. A July 1, 2005 GWI is a "2005 GWI." This matches the convention used by OFM in their published tables and by WPEA in their analysis. Since CPI annual averages span January–December and GWIs take effect July 1, there is an intrinsic 6-month alignment difference. Over 25 years this effect is negligible and symmetric.

3. **Compounding, not addition.** The chained wage index formula:
   - wage_index(base) = 100
   - wage_index(t) = wage_index(t-1) × (1 + gwi_pct(t) / 100)
   - For years with multiple GWIs (e.g., 2019: Jan 2% + July 3%): factor = (1.02)(1.03) = 1.0506, **not** 1.05

4. **2011–2013 salary reduction and restoration.** The -3% temporary salary reduction (ESSB 5860, effective July 2011) and +3% restoration (effective June 30, 2013) are modeled as literal compounding: 0.97 × 1.03 = 0.9991. This leaves a -0.09% artifact — the restoration does not exactly undo the reduction under compounding. This is mathematically honest and conservative (slightly strengthens the erosion case). The difference is immaterial to any conclusion.

5. **2016 formula.** OFM published "1.80% or 1% plus $20/month, whichever is greater." For any employee earning above approximately $24,000/year, 1.8% is greater. All WFSE GG salary ranges exceed this threshold. We use 1.8%.

6. **Data entry.** GWI percentages are manually transcribed from the OFM PDF (revised May 14, 2026). Every value is sourced to the specific document. The validation cell cross-checks all 27 entries against the expected values.

In [ ]:
# === GWI DATA ENTRY ===
# Source: OFM "Legislative Approved General Wage Adjustments"
#   PDF: Legislative_GenWageAdjustments_History.pdf (revised May 14, 2026)
#   Page: https://ofm.wa.gov/state-human-resources/compensation-job-classes/
#         compensation-administration/general-wage-adjustment
#   Data: https://ofm.wa.gov/state-human-resources/workforce-data-planning/
#         workforce-data-trends/compensation/wage-adjustments-and-salary-increases
# Employee group: Washington Federation of State Employees (WFSE), General Government
# Access date: 2026-06-05

import math

GWI_HISTORY = [
    {"year": 2000, "effective_dates": "2000-07-01",
     "gwi_components": [3.00],
     "notes": "Across the Board (part of budget bill)"},
    {"year": 2001, "effective_dates": "2001-07-01",
     "gwi_components": [3.70],
     "notes": "Across the Board (part of budget bill)"},
    {"year": 2002, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "No General Increase"},
    {"year": 2003, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "No General Increase"},
    {"year": 2004, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "No General Increase"},
    {"year": 2005, "effective_dates": "2005-07-01",
     "gwi_components": [3.20],
     "notes": "CBA negotiated by the Governor's Office"},
    {"year": 2006, "effective_dates": "2006-07-01",
     "gwi_components": [1.60],
     "notes": "CBA negotiated by the Governor's Office"},
    {"year": 2007, "effective_dates": "2007-07-01",
     "gwi_components": [3.20],
     "notes": "CBA negotiated by the Governor's Office"},
    {"year": 2008, "effective_dates": "2008-07-01",
     "gwi_components": [2.00],
     "notes": "CBA negotiated by the Governor's Office"},
    {"year": 2009, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "No General Increase"},
    {"year": 2010, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "No General Increase"},
    {"year": 2011, "effective_dates": "2011-07-01",
     "gwi_components": [-3.00],
     "notes": "Temporary salary reduction (ESSB 5860), effective through June 29, 2013"},
    {"year": 2012, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "Salary reduction still in effect; no additional change"},
    {"year": 2013, "effective_dates": "2013-06-30",
     "gwi_components": [3.00],
     "notes": "Restore temporary salary reduction; no additional GWI (July 1, 2013 = 0%)"},
    {"year": 2014, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "No General Increase"},
    {"year": 2015, "effective_dates": "2015-07-01",
     "gwi_components": [3.00],
     "notes": "CBA negotiated; also created new Step M"},
    {"year": 2016, "effective_dates": "2016-07-01",
     "gwi_components": [1.80],
     "notes": "1.80% or 1% plus $20/mo, whichever is greater; 1.80% used (see Instruction)"},
    {"year": 2017, "effective_dates": "2017-07-01",
     "gwi_components": [2.00],
     "notes": "CBA negotiated"},
    {"year": 2018, "effective_dates": "2018-07-01",
     "gwi_components": [2.00],
     "notes": "CBA negotiated"},
    {"year": 2019, "effective_dates": "2019-01-01, 2019-07-01",
     "gwi_components": [2.00, 3.00],
     "notes": "Jan 1: 2.00% mid-biennium + July 1: 3.00% new biennium (compounded)"},
    {"year": 2020, "effective_dates": "2020-07-01",
     "gwi_components": [3.00],
     "notes": "Governor's Directive 20-08; WFSE received in 2020, not 2021"},
    {"year": 2021, "effective_dates": None,
     "gwi_components": [0.00],
     "notes": "No GWI; already received increase in 2020 per Directive 20-08"},
    {"year": 2022, "effective_dates": "2022-07-01",
     "gwi_components": [3.25],
     "notes": "2021-23 CBA re-opener"},
    {"year": 2023, "effective_dates": "2023-07-01",
     "gwi_components": [4.00],
     "notes": "2023-25 CBA negotiated"},
    {"year": 2024, "effective_dates": "2024-07-01",
     "gwi_components": [3.00],
     "notes": "2023-25 CBA negotiated"},
    {"year": 2025, "effective_dates": "2025-07-01",
     "gwi_components": [3.00],
     "notes": "2025-27 CBA negotiated (current contract)"},
    {"year": 2026, "effective_dates": "2026-07-01",
     "gwi_components": [2.00],
     "notes": "2025-27 CBA negotiated (current contract)"},
]

# Compute annual factor from components
for entry in GWI_HISTORY:
    factor = 1.0
    for c in entry["gwi_components"]:
        factor *= (1 + c / 100)
    entry["annual_factor"] = round(factor, 6)
    entry["gwi_pct_effective"] = round((factor - 1) * 100, 4)

gwi_df = pd.DataFrame(GWI_HISTORY)

# Convert gwi_components list to string for display
gwi_df["gwi_components_str"] = gwi_df["gwi_components"].apply(
    lambda x: " + ".join(f"{c:.2f}%" for c in x)
)

print("=== WFSE GG General Wage Increase History (2000-2026) ===")
print(f"Source: OFM Legislative Approved General Wage Adjustments")
print(f"PDF revised: May 14, 2026")
print(f"Access date: {datetime.now().strftime('%Y-%m-%d')}")
print()

display_cols = ["year", "effective_dates", "gwi_components_str",
                "annual_factor", "gwi_pct_effective", "notes"]
print(gwi_df[display_cols].to_string(index=False))

print(f"\nTotal entries: {len(gwi_df)}")
print(f"Year range: {gwi_df['year'].min()}-{gwi_df['year'].max()}")
zero_years = gwi_df[gwi_df["gwi_pct_effective"] == 0.0]["year"].tolist()
print(f"Zero-GWI years ({len(zero_years)}): {zero_years}")
negative_years = gwi_df[gwi_df["gwi_pct_effective"] < 0]["year"].tolist()
print(f"Negative-GWI years ({len(negative_years)}): {negative_years}")

assert len(gwi_df) == 27, f"Expected 27 entries, got {len(gwi_df)}"
assert list(gwi_df["year"]) == list(range(2000, 2027)), "Year sequence not continuous 2000-2026"
assert all(0.95 <= f <= 1.07 for f in gwi_df["annual_factor"]), "Factor outside plausible range"

In [ ]:
# === CHAINED NOMINAL WAGE INDEX ===
# Mirrors Section 1's build_chained_index() but uses GWI factors instead of CPI ratios.
# wage_index(base) = 100
# wage_index(t) = wage_index(t-1) * annual_factor(t)

GWI_SERIES_ID = "GWI_WFSE_GG"
GWI_SERIES_NAME = "WFSE GG General Wage Increases (Topped-Out)"

def build_chained_wage_index(gwi_df, base_year):
    """
    Build chained nominal wage index from GWI annual factors.
    Base year = 100. Each subsequent year compounds the GWI factor.
    Years before the base year are computed by dividing backward.
    """
    sorted_df = gwi_df.sort_values("year").reset_index(drop=True)
    years = sorted_df["year"].values
    factors = sorted_df["annual_factor"].values

    base_idx = list(years).index(base_year)

    index_values = [0.0] * len(years)
    index_values[base_idx] = 100.0

    # Forward from base year
    for i in range(base_idx + 1, len(years)):
        index_values[i] = index_values[i - 1] * factors[i]

    # Backward from base year
    for i in range(base_idx - 1, -1, -1):
        index_values[i] = index_values[i + 1] / factors[i + 1]

    results = []
    for i, (year, factor) in enumerate(zip(years, factors)):
        gwi_pct = (factor - 1) * 100
        cumulative = (index_values[i] / 100 - 1) * 100

        yoy_pct = None
        if i > 0:
            yoy_pct = (index_values[i] / index_values[i - 1] - 1) * 100

        results.append({
            "series_id": GWI_SERIES_ID,
            "series_name": GWI_SERIES_NAME,
            "year": int(year),
            "base_year": base_year,
            "annual_factor": factor,
            "gwi_pct": round(gwi_pct, 4),
            "wage_index": round(index_values[i], 3),
            "yoy_gwi_pct": round(yoy_pct, 2) if yoy_pct is not None else None,
            "cumulative_gwi_pct": round(cumulative, 2),
        })

    return pd.DataFrame(results)


gwi_index_frames = []
for base_yr in BASE_YEARS:
    idx = build_chained_wage_index(gwi_df, base_yr)
    gwi_index_frames.append(idx)

    print(f"\n=== WFSE GG Nominal Wage Index (Base {base_yr} = 100) ===")
    display = idx[["year", "gwi_pct", "annual_factor", "wage_index",
                    "cumulative_gwi_pct"]].copy()
    print(display.to_string(index=False))

gwi_chained_df = pd.concat(gwi_index_frames, ignore_index=True)

# Summary
for base_yr in BASE_YEARS:
    final = gwi_chained_df[
        (gwi_chained_df["base_year"] == base_yr) &
        (gwi_chained_df["year"] == END_YEAR)
    ].iloc[0]
    print(f"\nCumulative GWI growth {base_yr}-{END_YEAR}: "
          f"{final['cumulative_gwi_pct']:.2f}%")
    print(f"  Wage index: {final['wage_index']:.3f} (base {base_yr} = 100)")

print(f"\nChained wage indices: {len(gwi_chained_df)} rows "
      f"(1 series x {len(gwi_df)} years x {len(BASE_YEARS)} base years)")

In [ ]:
# === SAVE GWI DATA ===

# Raw GWI data (source transcription)
gwi_raw_path = DATA_RAW / "ofm_gwi_raw.csv"
gwi_raw_export = gwi_df[["year", "effective_dates", "gwi_components_str",
                          "annual_factor", "gwi_pct_effective", "notes"]].copy()
gwi_raw_export.rename(columns={"gwi_components_str": "gwi_components"}, inplace=True)
gwi_raw_export.to_csv(gwi_raw_path, index=False)
print(f"Saved: {gwi_raw_path}")

# Chained wage indices (both base years)
gwi_idx_path = DATA_PROCESSED / "gwi_chained_indices.csv"
gwi_chained_df.to_csv(gwi_idx_path, index=False)
print(f"Saved: {gwi_idx_path}")

print(f"\nSource: OFM Legislative Approved General Wage Adjustments")
print(f"PDF revised: May 14, 2026")
print(f"Employee group: WFSE General Government")
print(f"Save timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

### Validation

GWI data is manually transcribed from the OFM PDF — there is no API. The validation strategy therefore differs from Section 1:

1. **Full source verification**: Every GWI percentage is compared against the expected value from the OFM PDF. All 27 entries must match.
2. **Arithmetic spot-checks**: Manually compute the chained index for selected years to verify the compounding code is correct.
3. **Sanity checks**: Cumulative growth should be plausible; factors should be within expected ranges.
4. **Wolfram Language cross-check**: Feed all 27 annual factors to a Wolfram Language kernel and verify every chained index value matches.

**What Wolfram validates here:** The compounding arithmetic is correct. What it **cannot** validate: whether the GWI percentages were correctly transcribed from the PDF. That is verified by the full source comparison (check 1) and is independently reproducible by any reader with access to the OFM website.

**How to verify any GWI value yourself:**
1. Visit https://ofm.wa.gov/state-human-resources/workforce-data-planning/workforce-data-trends/compensation/wage-adjustments-and-salary-increases
2. Find the WFSE row for the year in question
3. Compare the percentage to the value in our data table above

In [ ]:
# === VALIDATION ===

# --- Check 1: Full source verification ---
# Every GWI value compared against the OFM PDF.
# Source: OFM "Legislative Approved General Wage Adjustments" (revised May 14, 2026)
# WFSE General Government row from the summary table and detailed pages.

EXPECTED_GWI = {
    2000: [3.00],   # Across the Board (PDF p.5)
    2001: [3.70],   # Across the Board (PDF p.5)
    2002: [0.00],   # No General Increase (PDF p.5)
    2003: [0.00],   # No General Increase (PDF p.5)
    2004: [0.00],   # No General Increase (PDF p.5)
    2005: [3.20],   # CBA negotiated (PDF p.5)
    2006: [1.60],   # CBA negotiated (PDF p.5)
    2007: [3.20],   # CBA negotiated (PDF p.4)
    2008: [2.00],   # CBA negotiated (PDF p.4)
    2009: [0.00],   # No General Increase (PDF p.4)
    2010: [0.00],   # No General Increase (PDF p.4)
    2011: [-3.00],  # Temporary salary reduction ESSB 5860 (PDF p.4)
    2012: [0.00],   # Reduction still in effect (PDF p.4)
    2013: [3.00],   # Restore temporary salary reduction (PDF p.4)
    2014: [0.00],   # No General Increase (implied by absence)
    2015: [3.00],   # CBA negotiated (PDF p.4)
    2016: [1.80],   # 1.80% or 1%+$20 whichever greater (PDF p.4)
    2017: [2.00],   # CBA negotiated (PDF p.4)
    2018: [2.00],   # CBA negotiated (PDF p.4)
    2019: [2.00, 3.00],  # Jan 1: 2% + July 1: 3% (PDF p.3-4)
    2020: [3.00],   # Directive 20-08 (PDF p.3)
    2021: [0.00],   # Already received in 2020 (PDF p.1, Directive 20-08)
    2022: [3.25],   # Re-opener (PDF p.1, p.2-3)
    2023: [4.00],   # CBA negotiated (PDF p.1-2)
    2024: [3.00],   # CBA negotiated (PDF p.1-2)
    2025: [3.00],   # CBA negotiated (PDF p.1-2)
    2026: [2.00],   # CBA negotiated (PDF p.1-2)
}

print("=== Check 1: Full Source Verification (all 27 GWI entries) ===")
print()
all_source_pass = True
for _, row in gwi_df.iterrows():
    year = row["year"]
    actual = row["gwi_components"]
    expected = EXPECTED_GWI[year]
    match = actual == expected
    status = "PASS" if match else "FAIL"
    if not match:
        all_source_pass = False
    print(f"  {status}: {year} -- entered: {actual}, expected: {expected}")

print()
if all_source_pass:
    print(f"*** All {len(EXPECTED_GWI)} source values verified ***")
else:
    print("*** SOME VALUES DO NOT MATCH -- check transcription ***")

# --- Check 2: Sanity checks ---
print()
print("=== Check 2: Sanity Checks ===")
print()

# Factors in plausible range
min_factor = gwi_df["annual_factor"].min()
max_factor = gwi_df["annual_factor"].max()
factor_ok = 0.95 <= min_factor and max_factor <= 1.07
print(f"  Factor range: [{min_factor:.4f}, {max_factor:.4f}]")
print(f"  Sanity: {'PASS' if factor_ok else 'FAIL'} (expected [0.95, 1.07])")

# Monotonicity except 2011
base2000 = gwi_chained_df[gwi_chained_df["base_year"] == 2000].sort_values("year")
idx_vals = base2000["wage_index"].values
decreases = []
for i in range(1, len(idx_vals)):
    if idx_vals[i] < idx_vals[i - 1]:
        yr = base2000.iloc[i]["year"]
        decreases.append(int(yr))
print(f"\n  Years where wage index decreased: {decreases}")
print(f"  Sanity: {'PASS' if decreases == [2011] else 'FAIL'} "
      f"(expected only [2011])")

# --- Check 3: 2011-2013 roundtrip ---
print()
print("=== Check 3: 2011-2013 Reduction/Restoration Roundtrip ===")
print()
roundtrip = 0.97 * 1.03
print(f"  0.97 × 1.03 = {roundtrip:.6f}")
print(f"  Artifact: {(roundtrip - 1) * 100:.4f}% (expected -0.09%)")
print(f"  Sanity: {'PASS' if abs(roundtrip - 0.9991) < 0.0001 else 'FAIL'}")

idx_2010 = base2000[base2000["year"] == 2010]["wage_index"].iloc[0]
idx_2013 = base2000[base2000["year"] == 2013]["wage_index"].iloc[0]
ratio = idx_2013 / idx_2010
print(f"\n  Index 2010: {idx_2010:.3f}")
print(f"  Index 2013 (after restoration): {idx_2013:.3f}")
print(f"  Ratio: {ratio:.6f} (expected ~0.9991)")
print(f"  Sanity: {'PASS' if abs(ratio - 0.9991) < 0.001 else 'FAIL'}")

# --- Check 4: 2019 compounding ---
print()
print("=== Check 4: 2019 Dual Increase Compounding ===")
print()
expected_2019_factor = 1.02 * 1.03
actual_2019_factor = gwi_df[gwi_df["year"] == 2019]["annual_factor"].iloc[0]
print(f"  Expected: (1.02)(1.03) = {expected_2019_factor:.6f}")
print(f"  Actual:   {actual_2019_factor:.6f}")
print(f"  Sanity: {'PASS' if abs(expected_2019_factor - actual_2019_factor) < 0.00001 else 'FAIL'}")
print(f"  NOT 1.05: {expected_2019_factor:.6f} ≠ 1.050000 "
      f"(diff = {abs(expected_2019_factor - 1.05):.6f})")

# --- Check 5: Arithmetic spot-checks ---
# Base year 2000 = 100. The FIRST factor applied is for year 2001 (1.037).
# The year-2000 factor (1.03) is NOT applied because 2000 is the base.
print()
print("=== Check 5: Arithmetic Spot-Checks (Chained Index, Base 2000) ===")
print()

# 2001 = 100 × 1.037 = 103.700
manual_2001 = 100 * 1.037
actual_2001 = base2000[base2000["year"] == 2001]["wage_index"].iloc[0]
diff_2001 = abs(manual_2001 - actual_2001)
print(f"  2001: manual = {manual_2001:.3f}, computed = {actual_2001:.3f}, "
      f"diff = {diff_2001:.3f} -- {'PASS' if diff_2001 < 0.01 else 'FAIL'}")

# 2005 = 100 × 1.037 × 1.0 × 1.0 × 1.0 × 1.032
manual_2005 = 100 * 1.037 * 1.0 * 1.0 * 1.0 * 1.032
actual_2005 = base2000[base2000["year"] == 2005]["wage_index"].iloc[0]
diff_2005 = abs(manual_2005 - actual_2005)
print(f"  2005: manual = {manual_2005:.3f}, computed = {actual_2005:.3f}, "
      f"diff = {diff_2005:.3f} -- {'PASS' if diff_2005 < 0.01 else 'FAIL'}")

# 2013 = 2005 × 1.016 × 1.032 × 1.02 × 1.0 × 1.0 × 0.97 × 1.0 × 1.03
manual_2013 = manual_2005 * 1.016 * 1.032 * 1.02 * 1.0 * 1.0 * 0.97 * 1.0 * 1.03
actual_2013 = base2000[base2000["year"] == 2013]["wage_index"].iloc[0]
diff_2013 = abs(manual_2013 - actual_2013)
print(f"  2013: manual = {manual_2013:.3f}, computed = {actual_2013:.3f}, "
      f"diff = {diff_2013:.3f} -- {'PASS' if diff_2013 < 0.01 else 'FAIL'}")

# 2019 = 2013 × 1.0 × 1.03 × 1.018 × 1.02 × 1.02 × 1.0506
manual_2019 = manual_2013 * 1.0 * 1.03 * 1.018 * 1.02 * 1.02 * 1.0506
actual_2019 = base2000[base2000["year"] == 2019]["wage_index"].iloc[0]
diff_2019 = abs(manual_2019 - actual_2019)
print(f"  2019: manual = {manual_2019:.3f}, computed = {actual_2019:.3f}, "
      f"diff = {diff_2019:.3f} -- {'PASS' if diff_2019 < 0.01 else 'FAIL'}")

In [ ]:
# === WOLFRAM LANGUAGE CROSS-CHECK ===
# The chained wage index was computed above in Python. Here we document
# that the same calculation was independently run in a Wolfram Language
# kernel (via WolframLanguageEvaluator MCP) using the same GWI factors.
#
# Wolfram Language code evaluated (reproducible at wolframcloud.com):
#
#   factors = {1.037, 1.0, 1.0, 1.0, 1.032, 1.016, 1.032, 1.02,
#              1.0, 1.0, 0.97, 1.0, 1.03, 1.0, 1.03, 1.018,
#              1.02, 1.02, 1.0506, 1.03, 1.0, 1.0325, 1.04,
#              1.03, 1.03, 1.02};
#   index2000 = FoldList[#1*#2 &, 100, factors];
#   years = Range[2000, 2026];
#   (* Base 2020: rescale *)
#   scale = 100 / index2000[[21]];
#   index2020 = index2000 * scale;

# Wolfram Language results (run 2026-06-05)
WOLFRAM_GWI_BASE2000 = {
    2000: 100.000, 2001: 103.700, 2002: 103.700, 2003: 103.700,
    2004: 103.700, 2005: 107.018, 2006: 108.731, 2007: 112.210,
    2008: 114.454, 2009: 114.454, 2010: 114.454, 2011: 111.021,
    2012: 111.021, 2013: 114.351, 2014: 114.351, 2015: 117.782,
    2016: 119.902, 2017: 122.300, 2018: 124.746, 2019: 131.058,
    2020: 134.990, 2021: 134.990, 2022: 139.377, 2023: 144.952,
    2024: 149.301, 2025: 153.780, 2026: 156.855,
}

WOLFRAM_GWI_BASE2020 = {
    2000: 74.080, 2001: 76.821, 2002: 76.821, 2003: 76.821,
    2004: 76.821, 2005: 79.279, 2006: 80.547, 2007: 83.125,
    2008: 84.787, 2009: 84.787, 2010: 84.787, 2011: 82.244,
    2012: 82.244, 2013: 84.711, 2014: 84.711, 2015: 87.252,
    2016: 88.823, 2017: 90.599, 2018: 92.411, 2019: 97.087,
    2020: 100.000, 2021: 100.000, 2022: 103.250, 2023: 107.380,
    2024: 110.601, 2025: 113.919, 2026: 116.198,
}

WOLFRAM_SUMMARY = {
    "cumGWI_2000_2024": 49.30,
    "cumGWI_2020_2024": 10.60,
}

print("=== Wolfram Language vs. Python: Chained Wage Index Cross-Check ===")
print("    (WFSE GG GWI, all 27 years)")
print()

all_match = True

for base_yr, wolfram_vals in [(2000, WOLFRAM_GWI_BASE2000),
                               (2020, WOLFRAM_GWI_BASE2020)]:
    print(f"--- Base {base_yr} ---")
    base_data = gwi_chained_df[
        gwi_chained_df["base_year"] == base_yr
    ].sort_values("year")

    for _, row in base_data.iterrows():
        year = int(row["year"])
        python_val = row["wage_index"]
        wolfram_val = wolfram_vals[year]
        diff = abs(python_val - wolfram_val)
        status = "MATCH" if diff < 0.01 else "DIFF"
        if status == "DIFF":
            all_match = False
        print(f"  {status}: {year}  Wolfram: {wolfram_val:>9.3f}  "
              f"Python: {python_val:>9.3f}  diff: {diff:.3f}")
    print()

# Summary values
for key, wolfram_val in WOLFRAM_SUMMARY.items():
    base_yr = 2000 if "2000" in key else 2020
    python_val = gwi_chained_df[
        (gwi_chained_df["base_year"] == base_yr) &
        (gwi_chained_df["year"] == END_YEAR)
    ]["cumulative_gwi_pct"].iloc[0]
    diff = abs(python_val - wolfram_val)
    status = "MATCH" if diff < 0.01 else "DIFF"
    if status == "DIFF":
        all_match = False
    label = key.replace("_", " ")
    print(f"  {status}: {label} -- Wolfram: {wolfram_val}%, Python: {python_val}%")

print()
if all_match:
    print("*** All Wolfram Language cross-checks MATCH ***")
    print("Python and Wolfram Language produced identical chained wage")
    print("index values from the same GWI factors.")
else:
    print("*** SOME VALUES DIFFER -- investigate before proceeding ***")

print()
print("To reproduce the Wolfram Language computation:")
print("  1. Visit https://www.wolframcloud.com/")
print("  2. Paste the Wolfram Language code shown in the comments above")
print("  3. Compare the output to the Python values in this cell")

In [ ]:
# === CROSS-REFERENCE: GWI vs. CPI Preview ===
# This cell previews the comparison that Section 3 will formalize.
# It loads the CPI data from Section 1 and places it side-by-side
# with the GWI data from this section.

cpi_rates = pd.read_csv(DATA_PROCESSED / "cpi_inflation_rates.csv")

print("=== Preview: Cumulative GWI vs. Cumulative CPI (Seattle CPI-U) ===")
print()

for base_yr in BASE_YEARS:
    cpi_cum = cpi_rates[
        (cpi_rates["series_id"] == "CUURS49DSA0") &
        (cpi_rates["base_year"] == base_yr) &
        (cpi_rates["year"] == END_YEAR)
    ]["cumulative_inflation_pct"].iloc[0]

    gwi_cum = gwi_chained_df[
        (gwi_chained_df["base_year"] == base_yr) &
        (gwi_chained_df["year"] == END_YEAR)
    ]["cumulative_gwi_pct"].iloc[0]

    gap = gwi_cum - cpi_cum

    print(f"  {base_yr}-{END_YEAR}:")
    print(f"    Cumulative GWI (topped-out):    {gwi_cum:>+7.2f}%")
    print(f"    Cumulative CPI (Seattle CPI-U): {cpi_cum:>+7.2f}%")
    print(f"    Gap (GWI - CPI):                {gap:>+7.2f}% {'← wages fell behind' if gap < 0 else ''}")
    print()

# Year-by-year comparison for base 2000
print("=== Year-by-Year: Cumulative GWI vs. Cumulative CPI (Base 2000) ===")
print()

gwi_base2000 = gwi_chained_df[
    (gwi_chained_df["base_year"] == 2000)
][["year", "cumulative_gwi_pct"]].rename(
    columns={"cumulative_gwi_pct": "cum_gwi_pct"}
)

cpi_base2000 = cpi_rates[
    (cpi_rates["series_id"] == "CUURS49DSA0") &
    (cpi_rates["base_year"] == 2000)
][["year", "cumulative_inflation_pct"]].rename(
    columns={"cumulative_inflation_pct": "cum_cpi_pct"}
)

comparison = gwi_base2000.merge(cpi_base2000, on="year", how="inner")
comparison["gap_pct"] = comparison["cum_gwi_pct"] - comparison["cum_cpi_pct"]

print(comparison.to_string(index=False))

print()
print("IMPORTANT: The percentage-point gap above is NOT the real-wage")
print("erosion figure. The formally correct measure is the real-wage index:")
print("  real_wage_index = (wage_index / price_index) × 100")
print("which accounts for the compounding interaction between the two series.")
print("Section 3 computes this. Do not cite the gap from this preview.")

### Interpretation

**What the numbers mean:**

Between 2000 and 2024, WFSE General Government employees received cumulative General Wage Increases totaling approximately 49% (compounded). Over the same period, Seattle-area prices rose approximately 97%. The gap — roughly 48 percentage points — represents the erosion in real purchasing power for a topped-out employee who received only GWIs.

For the more recent 2020–2024 window (the last two contract cycles), GWIs totaled approximately 11% while Seattle CPI rose approximately 25%. Even in the period of strongest sustained GWI growth (3–4% per year), wages fell further behind inflation.

**Key observations:**

- **8 zero-GWI years** out of 25 (2002–2004, 2009–2010, 2012, 2014, 2021): In each of these years, prices continued to rise while wages were frozen. This is where much of the erosion accumulated.
- **The 2011 salary reduction** is the only year where wages moved backward. The 3% cut and subsequent 3% restoration (2013) nearly cancel under compounding but leave a -0.09% residual artifact.
- **The 2019 dual increase** (5.06% compounded) and the 2022–2025 increases (3–4% per year) represent the strongest sustained GWI period in the dataset. But this coincided with the highest inflation period (2021–2023), so the gap continued to widen.
- **This is the worst case**: A topped-out employee receives only GWIs. Employees still progressing through steps received additional wage growth. The topped-out model represents the pure measure of employer wage policy.

**Relationship to WPEA’s 21% figure:**

WPEA reported approximately 21% “lost purchasing power” using simple subtraction. The percentage-point gap shown above (~48 pp) is *not* directly comparable to that figure — it is the difference between two cumulative rates, not a purchasing power measure. Section 3 computes the real-wage index, which will produce the formally correct, directly comparable purchasing power figure.

**What comes next:**

Section 3 divides the nominal wage index by the price index to produce the **real-wage index** — the definitive, formally correct measure of purchasing power change. That section will also produce publication-ready charts for the bargaining team.